# Workshop 2 (Advanced): LLM APIs, Prompting and RAG
**NUS SDS Datathon 2027** · Solution notebook

In this workshop you will build a small question-answering system, one step at a time:

**plain baseline → better prompting → retrieval → generation with citations → evaluation**

Our teaching domain is the **2026 Formula One season**. It happened after the language models we use were trained, so a plain model cannot know it and has to guess. Our knowledge source is Wikipedia ([CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)), in English, Malay and Chinese.

> This is a self-contained teaching example. It does not use any competition data.

## Setup

First, get a free SEA-LION API key at [playground.sea-lion.ai](https://playground.sea-lion.ai) → *API Key* → *Create New Trial API Key*. Copy it straight away; it is only shown once.

**Colab:**
1. Click the 🔑 **Secrets** icon in the left sidebar, add a secret named `SEALION_API_KEY` with your key as the value, and turn on **Notebook access**.
2. Run the cells from the top. The first cell installs the packages.

**VS Code / Other IDEs:**
1. Install [uv](https://docs.astral.sh/uv/getting-started/installation/) and the VS Code **Python** and **Jupyter** extensions.
2. Put this notebook in a folder, then run these in a terminal in that folder:
   ```
   uv venv --python 3.12 --seed
   uv pip install ipykernel
   ```
3. Create a file called `.env` in that folder containing `SEALION_API_KEY=sk-your-key-here`. Never share or commit this file.
4. Open the notebook → **Select Kernel** → `.venv`.
5. Run the cells from the top.

The key cell below checks where it is running: in Colab it reads the key from Secrets, otherwise from `.env`. Everything after that is the same.

In [ ]:
# Install the libraries this notebook needs (takes about a minute in Colab).
%pip install -q "openai==3.42.1" "python-dotenv==1.2.4"

In [ ]:
import os

# Read API keys from Colab Secrets when running in Colab, otherwise from a .env file.
# Never paste a key directly into a notebook: notebooks get shared, and so would your key.
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()


def get_secret(name):
    """Return the secret called `name`, or None if it has not been set."""
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:  # secret missing, or notebook access not switched on
            return None
    return os.getenv(name)


SEALION_API_KEY = get_secret("SEALION_API_KEY")

if SEALION_API_KEY:
    print(f"SEALION_API_KEY loaded (ends in ...{SEALION_API_KEY[-4:]})")
else:
    print("SEALION_API_KEY not found. Follow the setup steps above, then re-run this cell.")

---
## 0:00–0:10 Baseline demonstration

Language models answer fluently, even when they are wrong. That is a problem when you need answers people can trust.

Below are three questions about the 2026 F1 season, one each in English, Malay and Chinese. For each one we ask SEA-LION twice:

1. **Plain:** just the question.
2. **Grounded:** the question plus a short excerpt from Wikipedia, with an instruction to answer only from that excerpt and cite it.

Watch how confident the plain answers sound.

In [ ]:
from openai import OpenAI

SEALION_BASE_URL = "https://api.sea-lion.ai/v1"
SEALION_MODEL = "aisingapore/Qwen-SEA-LION-v4.5-27B-IT"

# SEA-LION speaks the same API as OpenAI, so we can use the `openai` library and just
# point it at SEA-LION's URL. ("missing-key" keeps this cell from crashing if no key is set.)
client = OpenAI(api_key=SEALION_API_KEY or "missing-key", base_url=SEALION_BASE_URL)


def ask(prompt):
    """Send one prompt to SEA-LION and return the text of its reply.

    This is the bare minimum. At 0:10 we will see why real code also needs
    retries, timeouts, caching and cost logging.
    """
    response = client.chat.completions.create(
        model=SEALION_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0, # makes the model pick its most likely words, so it gives roughly the same answer each run
    )
    return response.choices[0].message.content.strip()

### The demo questions

Each question comes with an excerpt copied from a **pinned revision** of a Wikipedia article. Pinning a revision means the text cannot change under us, so anyone can check exactly what the model was shown.

`DEMO_QUESTIONS` is a list of three dictionaries, one per question. Each holds:
- the question and its language,
- an excerpt copied from Wikipedia that contains the answer,
- the source title and link, pointing at an exact, frozen version of the article ("pinned revision"), so anyone can check the excerpt.

In [ ]:
# Each question is a dictionary. Not every field goes to the model:
#   question, excerpt, source_title -> sent to the model (excerpt and title only in the grounded call)
#   source_url, language            -> for humans: the link to check the source, and the table label
DEMO_QUESTIONS = [
    {
        "language": "English",
        "question": "In the 2026 Formula One rules, what replaced DRS, and when can a driver use it?",
        "source_title": "2026 Formula One World Championship (en.wikipedia.org)",
        "source_url": "https://en.wikipedia.org/w/index.php?oldid=1383335768",
        "excerpt": (
            "The concept saw the elimination of the drag reduction system (DRS), being replaced by a "
            "new overtake mode, initially referred to as manual override mode. [...] \"active "
            "aerodynamics\" (or simply \"active aero\") to describe the moveable front and rear wing "
            "elements – in both higher and lower drag/downforce configuration (called \"corner mode\" "
            "and \"straightline mode\" respectively); [...] and \"overtake mode\" to describe a mode "
            "where extra energy stored is used in order to help one driver overtake another, but only "
            "if the driver behind is within one second of the one ahead (similar to pre-2026 DRS "
            "deployment rules)."
        ),
    },
    {
        "language": "Malay",
        "question": "Siapakah yang membekalkan unit kuasa kepada pasukan Alpine dalam musim Formula Satu 2026?",
        "source_title": "Formula Satu musim 2026 (ms.wikipedia.org)",
        "source_url": "https://ms.wikipedia.org/w/index.php?oldid=6973858",
        "excerpt": (
            "Ford kembali ke sukan ini buat kali pertama sejak 2004, dengan menyokong Red Bull "
            "Powertrains dalam pembekalan unit kuasa kepada Red Bull Racing dan Racing Bulls. "
            "Renault tidak lagi menjadi pembekal enjin memandangkan Alpine telah beralih kepada "
            "penggunaan unit kuasa Mercedes."
        ),
    },
    {
        "language": "Chinese",
        "question": "2026年F1巴林大奖赛在哪个赛道举行？",
        "source_title": "2026年世界一级方程式锦标赛 (zh.wikipedia.org)",
        "source_url": "https://zh.wikipedia.org/w/index.php?oldid=95375520",
        "excerpt": (
            "2026年7月26日，国际汽车联盟正式确认巴林大奖赛将改在马来西亚雪邦赛道举行，"
            "并定名为“海湾航空巴林大奖赛马来西亚站”（Formula 1 Gulf Air Bahrain Grand Prix in Malaysia），"
            "举行日期安排在2026年10月2日至4日，这是马来西亚自2017赛季后相隔9年再次主办一级方程式的分站赛事。"
        ),
    },
]

In [ ]:
GROUNDED_TEMPLATE = """Answer the question using ONLY the source below.
Cite the source as [1] after each claim that uses it.
If the source does not contain the answer, say that you do not know.
Answer in the same language as the question.

Source [1]: {source_title}
{excerpt}

Question: {question}"""


def grounded_prompt(item):
    """Fill the template's three blanks. These are the only fields the model sees."""
    return GROUNDED_TEMPLATE.format(
        source_title=item["source_title"],
        excerpt=item["excerpt"],
        question=item["question"],
    )


print(grounded_prompt(DEMO_QUESTIONS[0]))

### Ask both ways

If the API is down or rate-limited during the live session, the cell below falls back to answers recorded earlier, and says so clearly.

In [ ]:
# Answers recorded on a successful run on 2026-11-02, used only if the live call fails.
RECORDED_ANSWERS = [
    {
        "plain": "As of the latest official FIA technical regulations and announcements leading into the 2026 season, **DRS (Drag Reduction System) is not being replaced**. The system remains a core part of Formula One's sporting and technical rules, though it may undergo minor adjustments to activation zones or criteria depending on circuit layout and FIA testing.\n\n### How DRS Works & When It Can Be Used (2026 Rules):\n- **Activation Condition**: A driver may activate DRS when they are **within 1.0 second** of the car ahead at a designated detection point (usually placed before a DRS zone).\n- **Where It Can Be Used**: Only in **officially designated DRS zones** on each circuit, which are typically long straights where overtaking is feasible.\n- **How It Works**: The rear wing's upper flap opens to reduce aerodynamic drag, increasing top speed by approximately 10–15 km/h.\n- **Restrictions**: DRS is disabled in wet conditions (when the FIA declares the track \"wet\"), during safety car periods, and if the driver is lapping slower traffic.\n\n### What's Actually Changing in 2026:\nThe 2026 regulations focus on:\n- New 1.6L V6 hybrid power units (100% sustainable fuels, higher electrical power output)\n- Simplified aerodynamics (new floor design, simpler front wings, removal of complex elements like bargeboards)\n- Enhanced safety standards and cost cap adjustments\n- **DRS is explicitly retained** in the FIA's 2026 Technical Regulations (Article 10.4), with no official replacement system announced.\n\n### Note on Rumors:\nOccasional speculation about \"DRS 2.0,\" variable activation, or replacement systems has circulated, but the FIA has consistently confirmed that DRS will remain, possibly with circuit-specific zone tweaks or minor timing adjustments based on 2024–2025 testing data.\n\nFor the most up-to-date circuit-specific DRS zones and any late regulatory tweaks, the FIA publishes official sporting regulations and circuit-specific appendices closer to each race weekend.",
        "grounded": "In the 2026 Formula One rules, the drag reduction system (DRS) was replaced by a new overtake mode, initially referred to as manual override mode [1]. A driver can use this mode only if they are within one second of the driver ahead [1]."
    },
    {
        "plain": "Setakat maklumat rasmi terkini, pasukan Alpine akan terus menggunakan unit kuasa daripada **Renault** (di bawah jenama Alpine) untuk musim Formula Satu 2026. \n\nWalaupun peraturan baharu 2026 memerlukan reka bentuk semula sepenuhnya bagi semua unit kuasa (termasuk peralihan kepada bahan api lestari dan perubahan besar pada sistem hibrid), tiada pengumuman rasmi daripada Alpine, Renault, atau FIA mengenai perubahan pembekal. Renault sedang membangunkan unit kuasa baharu yang mematuhi spesifikasi 2026 untuk digunakan oleh pasukan Alpine.\n\nPerlu diingat bahawa landskap pembekal unit kuasa dalam F1 boleh berubah jika terdapat pengumuman rasmi pada masa hadapan, tetapi setakat ini, **Renault kekal sebagai pembekal rasmi Alpine untuk musim 2026**.",
        "grounded": "Mercedes adalah pembekal unit kuasa kepada pasukan Alpine dalam musim Formula Satu 2026 [1]."
    },
    {
        "plain": "2026年F1巴林大奖赛预计仍将在**巴林国际赛道**（Bahrain International Circuit）举行。\n\n该赛道位于巴林王国的萨基尔（Sakhir），自2004年F1重返巴林以来，一直是巴林站的主办场地。目前F1官方尚未正式公布2026年完整赛历，但根据巴林方面与F1签订的长期主办合同以及历史惯例，赛道地点不会发生变更。如有任何官方调整，将以F1或巴林大奖赛组委会的正式公告为准。",
        "grounded": "2026年F1巴林大奖赛将在马来西亚雪邦赛道举行[1]。"
    }
]

In [ ]:
results = []
for i, item in enumerate(DEMO_QUESTIONS):
    try:
        # question alone, no grounding
        plain = ask(item["question"])

        # question + excerpt + instructions
        grounded = ask(grounded_prompt(item))
        origin = "live"
    except Exception as error:
        # If the API call fails, use the recorded answers from a previous successful run.
        print(f"Live call failed for question {i + 1} ({type(error).__name__}); using recorded answers.")
        plain, grounded = RECORDED_ANSWERS[i]["plain"], RECORDED_ANSWERS[i]["grounded"]
        origin = "recorded"
    results.append({**item, "plain": plain, "grounded": grounded, "origin": origin})

In [ ]:
from html import escape
from IPython.display import HTML, display


def show_side_by_side(results):
    rows = []
    for r in results:
        rows.append(f"""
        <tr><td colspan="2" style="background:#eef;padding:8px">
            <b>{escape(r['language'])}:</b> {escape(r['question'])}
            <span style="color:#888">({r['origin']})</span></td></tr>
        <tr>
            <td style="vertical-align:top;width:50%;padding:8px;white-space:pre-wrap">{escape(r['plain'])}</td>
            <td style="vertical-align:top;width:50%;padding:8px;white-space:pre-wrap">{escape(r['grounded'])}
                <br><br><small>[1] <a href="{r['source_url']}" target="_blank">{escape(r['source_title'])}</a></small></td>
        </tr>""")
    header = "<tr><th>Plain (no sources)</th><th>Grounded (with cited excerpt)</th></tr>"
    display(HTML(f"<table style='border-collapse:collapse;text-align:left'>{header}{''.join(rows)}</table>"))


show_side_by_side(results)

### "Why doesn't the model just search the web, like ChatGPT?"

**A model on its own cannot browse.** It only produces text from what it learned in training. SEA-LION, GPT and Gemini are all like this.

**ChatGPT is an app built around a model.** When a question needs fresh information, the app runs a web search, pastes the top results into the prompt, and asks the model to answer from them with citations. That is the same pattern as our grounded answers above: retrieve text, put it in the prompt, cite it. This pattern is called **RAG** (retrieval-augmented generation), with the web as the knowledge source.

**When we call SEA-LION's API, we get only the model.** There is no app around it, so nothing searches for it. Whatever retrieval we want, we build ourselves. That is what the RAG sections of this workshop teach.

**Why build our own retrieval instead of using web search?**
- **Control over sources:** we choose what the model reads (licensed, pinned Wikipedia revisions), not whatever a search engine returns today.
- **Reproducibility:** web results change daily, so the same question could get a different answer tomorrow. Pinned sources give the same input every run, which fair evaluation needs.
- **Traceable citations:** every `[1]` points to an exact text anyone can check.

### What you will build today, and why

The grounded answers above only worked because *we* hand-picked the right excerpt. Today you will build a system that does this automatically:

| Time | Step | Question it answers |
|---|---|---|
| 0:25 | **Baseline** | How good is a plain model on its own? |
| 0:40 | **Evaluate** | How do we measure "good" fairly? |
| 0:50 | **Prompt** | How much do better instructions help? |
| 1:00–1:30 | **Retrieve** | How do we find the right excerpt automatically? |
| 1:30 | **Generate with citations** | Can every claim be traced to a source? |
| 1:45 | **Error analysis** | When it fails, why? |

Three properties matter throughout:

- **Accuracy:** the facts are correct.
- **Appropriateness:** the answer suits the person asking, including answering in their language.
- **Traceable citations:** a reader can check every claim against its source. A fluent answer you cannot check is only a guess.